# Task 6: Visualization of Metrics and Graphs
Display all types of graphs associated with performance metrics.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, roc_curve, roc_auc_score,
                             precision_recall_curve)
import joblib
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')

In [ ]:
df = pd.read_csv('cardio_cleaned.csv')
X = df.drop('cardio', axis=1)
y = df['cardio']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = joblib.load('scaler.pkl')
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

model_files = {
    'Linear Regression': 'model_linear_regression.pkl',
    'Logistic Regression': 'model_logistic_regression.pkl',
    'SVM': 'model_svm.pkl',
    'KNN': 'model_knn.pkl',
    'Decision Tree': 'model_decision_tree.pkl',
    'Random Forest': 'model_random_forest.pkl'
}

models = {name: joblib.load(path) for name, path in model_files.items()}
print(f"Loaded {len(models)} models.")

In [ ]:
metrics_data = []
for name, model in models.items():
    if name == 'Linear Regression':
        y_pred = (model.predict(X_test_scaled) >= 0.5).astype(int)
    else:
        y_pred = model.predict(X_test_scaled)

    metrics_data.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1 Score': f1_score(y_test, y_pred)
    })

metrics_df = pd.DataFrame(metrics_data)
metrics_df

## 6.1 Grouped Bar Chart — All Metrics

In [ ]:
fig = go.Figure()
colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']

for idx, metric in enumerate(['Accuracy', 'Precision', 'Recall', 'F1 Score']):
    fig.add_trace(go.Bar(
        name=metric,
        x=metrics_df['Model'],
        y=metrics_df[metric],
        marker_color=colors[idx],
        text=metrics_df[metric].round(3),
        textposition='auto'
    ))

fig.update_layout(
    title='Model Performance Comparison — All Metrics',
    barmode='group', yaxis_title='Score',
    yaxis_range=[0.5, 0.85],
    template='plotly_white', height=500
)
fig.show()

## 6.2 Radar Chart — Model Comparison

In [ ]:
categories = ['Accuracy', 'Precision', 'Recall', 'F1 Score']

fig = go.Figure()
radar_colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6', '#1abc9c']

for idx, (_, row) in enumerate(metrics_df.iterrows()):
    values = [row[c] for c in categories] + [row[categories[0]]]
    fig.add_trace(go.Scatterpolar(
        r=values,
        theta=categories + [categories[0]],
        fill='toself',
        name=row['Model'],
        line_color=radar_colors[idx],
        opacity=0.6
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0.5, 0.85])),
    title='Radar Chart — Model Performance',
    template='plotly_white', height=600
)
fig.show()

## 6.3 ROC Curves (Plotly)

In [ ]:
fig = go.Figure()
roc_colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6', '#1abc9c']

for idx, (name, model) in enumerate(models.items()):
    if name == 'Linear Regression':
        y_scores = model.predict(X_test_scaled)
    elif hasattr(model, 'predict_proba'):
        y_scores = model.predict_proba(X_test_scaled)[:, 1]
    else:
        continue

    fpr, tpr, _ = roc_curve(y_test, y_scores)
    auc = roc_auc_score(y_test, y_scores)
    fig.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines',
                             name=f'{name} (AUC={auc:.3f})',
                             line=dict(color=roc_colors[idx], width=2)))

fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines',
                         line=dict(dash='dash', color='gray'), name='Random'))

fig.update_layout(
    title='ROC Curves — All Models',
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate',
    template='plotly_white', height=600
)
fig.show()

## 6.4 Precision-Recall Curves

In [ ]:
fig = go.Figure()

for idx, (name, model) in enumerate(models.items()):
    if name == 'Linear Regression':
        y_scores = model.predict(X_test_scaled)
    elif hasattr(model, 'predict_proba'):
        y_scores = model.predict_proba(X_test_scaled)[:, 1]
    else:
        continue

    precision, recall, _ = precision_recall_curve(y_test, y_scores)
    fig.add_trace(go.Scatter(x=recall, y=precision, mode='lines',
                             name=name, line=dict(color=roc_colors[idx], width=2)))

fig.update_layout(
    title='Precision-Recall Curves',
    xaxis_title='Recall', yaxis_title='Precision',
    template='plotly_white', height=600
)
fig.show()

## 6.5 Confusion Matrices (Heatmaps)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

for idx, (name, model) in enumerate(models.items()):
    row, col = idx // 3, idx % 3
    if name == 'Linear Regression':
        y_pred = (model.predict(X_test_scaled) >= 0.5).astype(int)
    else:
        y_pred = model.predict(X_test_scaled)

    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrRd', ax=axes[row][col],
                xticklabels=['Healthy', 'Disease'],
                yticklabels=['Healthy', 'Disease'])
    axes[row][col].set_title(f'{name}', fontsize=13, fontweight='bold')
    axes[row][col].set_ylabel('Actual')
    axes[row][col].set_xlabel('Predicted')

plt.suptitle('Confusion Matrices — All Models', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 6.6 Feature Importance (Random Forest)

In [ ]:
rf = models['Random Forest']
feat_imp = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=True)

fig = px.bar(feat_imp, x='Importance', y='Feature', orientation='h',
             title='Feature Importance — Random Forest',
             color='Importance', color_continuous_scale='RdYlGn')
fig.update_layout(template='plotly_white', height=500)
fig.show()

## 6.7 Final Accuracy Comparison Bar Chart

In [ ]:
sorted_df = metrics_df.sort_values('Accuracy', ascending=True)

fig = px.bar(sorted_df, x='Accuracy', y='Model', orientation='h',
             title='Model Accuracy Comparison',
             text=sorted_df['Accuracy'].round(4),
             color='Accuracy', color_continuous_scale='Viridis')
fig.update_layout(template='plotly_white', height=400, xaxis_range=[0.5, 0.85])
fig.update_traces(textposition='outside')
fig.show()